# CHSE Phase 1 — Two-Player Benchmark

This notebook walks through the complete two-player benchmark from Section 3 of the paper.

By the end you will have:

1. Verified the fixed-point formula and oscillation condition analytically
2. Reproduced Figure 2 (three HSI regimes)
3. Computed flip times and their dependence on HSI
4. Explored the phase portrait and stability scan

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from chse.core.primitives import Params, CapitalStocks, CHSEState
from chse.core.mechanisms import (
    ambiguity_push, reframe_resistance, reframe_success_prob,
    optimal_eta, optimal_kappa_spend,
)
from chse.benchmark.two_player import TwoPlayerModel
from chse.benchmark.oscillation import OscillationAnalysis
from chse.benchmark.flip_threshold import flip_time, flip_time_vs_hsi

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('All imports OK')

## 1. The hierarchy belief state

The primitive object is `CHSEState`, which holds the hierarchy belief `h_12 in [0,1]`.

- `h = 1.0` -> pure Stackelberg leadership (player 1 leads)
- `h = 0.5` -> full role ambiguity (Nash-like)
- `h = 0.0` -> pure followership

In [ ]:
s = CHSEState(h=0.75)
print(f'h_12 = {s.h}   (player 1 leads with probability 0.75)')
print(f'h_21 = {s.h_21}  (coherence: h_12 + h_21 = 1)')
print(f'Leader: player {s.leader()}')
print()
u_L, u_F = 10.0, 2.0
print(f'Player 1 payoff: {s.stage_payoff(u_L, u_F, 1):.2f}')
print(f'Player 2 payoff: {s.stage_payoff(u_L, u_F, 2):.2f}')

## 2. Hierarchy Stability Index and regime classification

In [ ]:
for hsi, label in [(2.1, 'stable'), (1.0, 'oscillatory'), (0.4, 'cascade')]:
    p = Params(HSI=hsi, PI=0.0)
    Z = p.instability_index(K_i=0, V_j=0)
    regime = p.regime(K_i=0, V_j=0)
    print(f'HSI={hsi:.1f}  Z={Z:.3f}  regime={regime!r}')

## 3. Mechanisms II and III

In [ ]:
p = Params(mu_II=1.0, lambda_kappa=1.0, lambda_R=1.5, alpha_R=0.4)

print('--- Mechanism II: Role Ambiguity ---')
for h in [0.3, 0.5, 0.7, 0.9]:
    delta = ambiguity_push(h=h, gamma=1.0, params=p)
    print(f'  h={h:.1f}  Delta_h = {delta:+.4f}')

print()
print('--- Mechanism III: Retroactive Reframing ---')
for eta in [0.5, 1.5, 3.0]:
    for rho in [0.0, 0.5, 0.9]:
        P_R = reframe_success_prob(eta=eta, rho=rho, params=p)
        print(f'  eta={eta:.1f} rho={rho:.1f}  P_R={P_R:.4f}')

## 4. Optimal best-response functions

From the paper's Bottleneck 2 derivation:

- `eta*(h)` = follower's optimal reframing spend (positive only when h < 0.5)
- `kappa*(h)` = leader's optimal resistance spend (positive only when h > 0.5)

In [ ]:
p = Params(alpha_R=0.4, lambda_R=1.5, lambda_kappa=1.0, c_mu=0.2, c_kappa=0.2)
h_vals = np.linspace(0.0, 1.0, 200)
eta_vals   = [optimal_eta(h, p) for h in h_vals]
kappa_vals = [optimal_kappa_spend(h, p) for h in h_vals]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(h_vals, eta_vals, color='#e05c3a', lw=2)
ax1.axvline(0.5, color='gray', ls='--', lw=0.8)
ax1.set_xlabel('h (hierarchy belief)')
ax1.set_ylabel('eta*(h)')
ax1.set_title('Optimal reframing spend (follower)')

ax2.plot(h_vals, kappa_vals, color='#2a7ab5', lw=2)
ax2.axvline(0.5, color='gray', ls='--', lw=0.8)
ax2.set_xlabel('h (hierarchy belief)')
ax2.set_ylabel('kappa*(h)')
ax2.set_title('Optimal resistance spend (leader)')

plt.tight_layout()
plt.show()

## 5. Oscillation condition

**Definition 3.1**: the system oscillates iff `mu^2 < 4 * eta_bar * kappa_bar`

The characteristic equation is `lambda^2 + mu*lambda + eta_bar*kappa_bar = 0`.

In [ ]:
for label, mu, eta, kappa in [
    ('Stable node   (HSI=2.1)', 2.0, 0.8, 0.2),
    ('Oscillatory   (HSI=1.0)', 0.6, 0.4, 0.4),
    ('Cascade       (HSI=0.4)', 0.3, 0.4, 0.4),
]:
    osc = OscillationAnalysis(mu=mu, eta_bar=eta, kappa_bar=kappa, r_bar=1.0)
    result = osc.analyse()
    print(label)
    print(result.summary())
    print()

## 6. Figure 2 — Three HSI regimes

Reproduces the three panels from Figure 2 of the paper.

In [ ]:
regimes = TwoPlayerModel.figure2_regimes(T=80)

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
colors = {'stable': '#2a7ab5', 'oscillatory': '#e8a020', 'cascade': '#c0392b'}
titles = {
    'stable':      'HSI = 2.1  (Stable)',
    'oscillatory': 'HSI = 1.0  (Oscillatory)',
    'cascade':     'HSI = 0.4  (Cascade)',
}

for ax, (name, result) in zip(axes, regimes.items()):
    t, h = result.t, result.h
    ax.fill_between(t, 0.5, h, where=(h >= 0.5), alpha=0.18, color=colors[name])
    ax.plot(t, h, color=colors[name], lw=1.4)
    ax.axhline(0.5, color='gray', ls='--', lw=0.8, alpha=0.7)
    ax.axhline(result.h_star, color=colors[name], ls=':', lw=1.0, alpha=0.6)
    ax.set_xlim(0, 80)
    ax.set_ylim(0, 1)
    ax.set_title(titles[name], fontsize=11)
    ax.set_xlabel('Period t')
    ax.text(2, 0.93, f'{result.turnover_count} flips', fontsize=9, color=colors[name])

axes[0].set_ylabel('h(t) — hierarchy belief')
plt.suptitle('Two-Player Benchmark: Hierarchy Belief Dynamics', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()
print('Shaded area = periods of player 1 leadership (h > 0.5).')

## 7. Fixed-point formula verification

In [ ]:
print('Fixed-point h* = 0.5 + (eta_bar - kappa_bar * r_bar) / (2 * mu)')
print()
for label, mu, eta, kappa, r in [
    ('stable',      2.0, 0.8, 0.2, 1.0),
    ('oscillatory', 0.6, 0.4, 0.4, 1.0),
    ('cascade',     0.3, 0.4, 0.4, 1.0),
]:
    model = TwoPlayerModel(mu=mu, eta_bar=eta, kappa_bar=kappa, r_bar=r)
    h_star_formula = 0.5 + (eta - kappa * r) / (2 * mu)
    h_star_method  = model.fixed_point()
    assert abs(h_star_formula - h_star_method) < 1e-12
    print(f'  {label}: h* = {h_star_method:.4f}  (formula verified OK)')

## 8. Leadership flip time t*

**Definition 3.2**: `t* = (1/mu_tilde) * ln((h0 - 0.5) / epsilon)`

Flip time is increasing in HSI and decreasing in reframing efficiency.

In [ ]:
result = flip_time(h0=0.75, mu=0.6, eta_bar=0.4, kappa_bar=0.4, r_bar=1.0, epsilon=0.01)
print('=== Single flip time example ===')
print(result.summary())
print()

hsi_vals = np.linspace(0.3, 2.5, 50)
_, t_stars = flip_time_vs_hsi(hsi_vals, h0=0.75, mu=0.8, eta_bar=0.4, r_bar=1.0)

fig, ax = plt.subplots(figsize=(7, 4))
valid = ~np.isnan(t_stars)
ax.plot(hsi_vals[valid], t_stars[valid], color='#2a7ab5', lw=2)
ax.axvline(1.0, color='gray', ls='--', lw=0.8, label='HSI = 1 (stability boundary)')
ax.set_xlabel('Hierarchy Stability Index (HSI)')
ax.set_ylabel('Flip time t*')
ax.set_title('Leadership flip time increases with HSI')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Stability scan over (mu, eta_bar) space

In [ ]:
mu_vals  = np.linspace(0.1, 3.0, 100)
eta_vals = np.linspace(0.05, 1.5, 100)
oscillates = OscillationAnalysis.stability_scan(mu_vals, eta_vals, kappa_bar=0.4, r_bar=1.0)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(eta_vals, mu_vals, oscillates.astype(float),
            levels=[0, 0.5, 1], colors=['#d4e8f7', '#f7d4d4'], alpha=0.8)
ax.contour(eta_vals, mu_vals, oscillates.astype(float), levels=[0.5], colors=['#555'])
ax.set_xlabel('eta_bar (reframing rate)')
ax.set_ylabel('mu (mean-reversion strength)')
ax.set_title('Oscillation condition: mu^2 < 4 * eta_bar * kappa_bar  (kappa_bar=0.4)')
ax.text(0.7, 2.5, 'Stable', fontsize=12, color='#1a5580')
ax.text(1.1, 0.5, 'Oscillatory', fontsize=12, color='#8b0000')
plt.tight_layout()
plt.show()

---

## Summary of Phase 1 verification

| Claim | Status |
|-------|--------|
| Fixed point formula h* = 0.5 + (eta_bar - kappa_bar*r_bar)/(2*mu) | OK |
| Oscillation condition mu^2 < 4*eta_bar*kappa_bar | OK |
| Flip time t* = (1/mu_tilde)*ln((h0-0.5)/epsilon) | OK |
| Stable regime: 0 flips, h converges to h* above 0.5 (HSI=2.1) | OK |
| Oscillatory regime: moderate flips (HSI=1.0) | OK |
| Cascade regime: many flips, wide amplitude (HSI=0.4) | OK |
| Optimal best-response functions eta*(h), kappa*(h) | OK |

**Next**: Phase 2 — full n-player dynamics, Mechanisms I & IV, phase diagram.